# TAMP-OS-Chroma: Tactile Colormap & Texture Lithography

**TAMP-OS-Chroma** es la extensión de **TAMP-OS** para convertir mapas de calor y gradientes científicos de color (ej. % de Litio en electrodos de $\text{Li}_x\text{Mn}_2\text{O}_4$, mapeos de microscopía óptica operando, EDS/SIMS) en **litografías 3D táctiles**.

### La Esencia de TAMP-OS: Rampa Continua + Textura Modulada
Para mantener la esencia de TAMP-OS y evitar la sobrecarga de zonas arbitrarias:
1. **Rampa macroscópica:** La elevación general de la pieza $Z_{\text{ramp}}(x, y)$ se eleva suavemente con el valor (pendiente continua reconocible por la cinestesia del brazo/dedo).
2. **Textura háptica modulada:** El usuario elige su textura preferida de la **Biblioteca de Texturas** (puntos, estrías, ondas, panal, escamas, etc.), y su **densidad espacial y relieve se saturan progresivamente** a medida que aumenta la concentración.
3. **Polaridad:** Se puede invertir para que mayor concentración sea la zona más lisa o la más saturada.


### 1. Importar librerías

In [ ]:
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LightSource
from PIL import Image

sys.path.insert(0, str(Path.cwd()))
from chroma_mapper import extract_concentration_map
from texture_engine import TEXTURE_LIBRARY, generate_textured_ramp
from stl_builder import heightmap_to_stl

print('TAMP-OS-Chroma listo. Texturas disponibles:', list(TEXTURE_LIBRARY.keys()))

### 2. Ver la Biblioteca de Texturas Táctiles

In [ ]:
cat_img = Image.open('examples/texture_library_catalog.png')
plt.figure(figsize=(10, 16))
plt.imshow(cat_img)
plt.axis('off')
plt.title('Biblioteca de Texturas Disponibles', fontsize=12, fontweight='bold')
plt.show()

### 3. Cargar imagen y decodificar concentración

In [ ]:
# Elige la imagen:
#   'examples/input_li_gradient.png' (Ejemplo 1: Gradiente Li / Inferno)
#   'examples/input_2d_spectrum.png'  (Ejemplo 2: Espectro 2D)
IMAGE_PATH = 'examples/input_li_gradient.png'
COLORMAP_METHOD = 'inferno'  # 'inferno', 'viridis', 'svd', 'turbo', o 'luminance'
INVERT = False               # True si quieres que mayor concentración = más liso

# Parámetros físicos (mm)
PRINT_WIDTH_MM = 100.0
BASE_THICKNESS_MM = 1.2
RAMP_HEIGHT_MM = 1.8         # Altura de la rampa macroscópica TAMP-OS
TEXTURE_RELIEF_MM = 0.6      # Relieve de la microtextura modulada

img = Image.open(IMAGE_PATH).convert('RGB')
aspect = img.height / img.width
PRINT_HEIGHT_MM = round(PRINT_WIDTH_MM * aspect, 2)
PHYSICAL_SIZE_MM = (PRINT_WIDTH_MM, PRINT_HEIGHT_MM)
GRID_RES = (400, int(400 * aspect))

scalar_map = extract_concentration_map(
    IMAGE_PATH,
    method=COLORMAP_METHOD,
    invert=INVERT,
    blur_sigma=1.0,
    target_size=GRID_RES
)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3))
ax1.imshow(img)
ax1.set_title('Imagen original en color')
ax1.axis('off')
im2 = ax2.imshow(scalar_map, cmap='viridis')
ax2.set_title('Mapa de Concentración [0, 1] (Base Rampa)')
ax2.axis('off')
plt.colorbar(im2, ax=ax2, fraction=0.046, pad=0.04)
plt.tight_layout()
plt.show()

### 4. Generar Rampa + Textura Modulada y Previsualizar

In [ ]:
# Elige tu textura: 'dots', 'lines_x', 'lines_y', 'diagonal_45', 'crosshatch', 'waves', 'honeycomb', 'scales'
SELECTED_TEXTURE = 'dots'

z_relief = generate_textured_ramp(
    scalar_map,
    texture_name=SELECTED_TEXTURE,
    physical_size_mm=PHYSICAL_SIZE_MM,
    ramp_height_mm=RAMP_HEIGHT_MM,
    texture_relief_mm=TEXTURE_RELIEF_MM
)

# Simulación de sombreado rasante (tacto)
ls = LightSource(azdeg=315, altdeg=35)
dx = PHYSICAL_SIZE_MM[0] / (GRID_RES[0] - 1)
dy = PHYSICAL_SIZE_MM[1] / (GRID_RES[1] - 1)
shaded = ls.hillshade(z_relief, vert_exag=2.0, dx=dx, dy=dy)

plt.figure(figsize=(10, 4.5))
plt.imshow(shaded, cmap='gray')
plt.title(f'Rampa TAMP-OS ({RAMP_HEIGHT_MM}mm) + Textura {SELECTED_TEXTURE} ({TEXTURE_RELIEF_MM}mm)', fontsize=11, fontweight='bold')
plt.axis('off')
plt.tight_layout()
plt.show()

### 5. Exportar archivo STL imprimible

In [ ]:
output_stl_path = f'examples/mi_litografia_rampa_{SELECTED_TEXTURE}.stl'
heightmap_to_stl(
    z_relief,
    output_stl_path,
    physical_size_mm=PHYSICAL_SIZE_MM,
    base_thickness_mm=BASE_THICKNESS_MM
)
print(f'¡Listo! STL exportado en: {output_stl_path}')